In [0]:
%sql
USE CATALOG iran_israel_capstone_project;
USE SCHEMA bronze;

In [0]:
landing_path = "/Volumes/iran_israel_capstone_project/bronze/landing_zone"

In [0]:
dbutils.fs.mkdirs(f"{landing_path}/market_data")
dbutils.fs.mkdirs(f"{landing_path}/events")

#ingest market_data to bronze layer


In [0]:
df_market_raw = spark.read.parquet("/Volumes/iran_israel_capstone_project/bronze/landing_zone/market_data/"
)
display(df_market_raw)
df_market_raw.printSchema()



writing to bronze layer finally


In [0]:
df_market_raw.write.mode("overwrite").saveAsTable("market_data")

#ingest events_timeline data to bronze layer

In [0]:
events_data = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/Volumes/iran_israel_capstone_project/bronze/landing_zone/events/")
display(events_data)
events_data.printSchema()

writing to bronze


In [0]:
events_data.write.mode("overwrite").saveAsTable("events")

#ingest alpha_vantage data to bronze layer


In [0]:
from pyspark.sql import functions as F

# Set schema
spark.sql("USE CATALOG iran_israel_capstone_project")
spark.sql("USE SCHEMA bronze")

# Landing zone paths
LANDING_PATH = "/Volumes/iran_israel_capstone_project/bronze/landing_zone/macro_data"

# Read data from landing zone
india_cpi_df = spark.read.parquet(f"{LANDING_PATH}/india_cpi")
wti_df = spark.read.parquet(f"{LANDING_PATH}/wti_crude")
brent_df = spark.read.parquet(f"{LANDING_PATH}/brent_crude_alpha")

# Standardization
def standardize_df(df, dataset_name):
    return (
        df
        .withColumnRenamed("date", "record_date")
        .withColumn("record_date", F.to_date("record_date"))
        .withColumn("dataset_name", F.lit(dataset_name))
        .withColumn("bronze_ingestion_time", F.current_timestamp())
    )

india_cpi_df = standardize_df(india_cpi_df, "india_cpi")
wti_df = standardize_df(wti_df, "wti_crude")
brent_df = standardize_df(brent_df, "brent_crude")

# Write Bronze tables
india_cpi_df.write.format("delta").mode("overwrite").saveAsTable("india_cpi_raw")

wti_df.write.format("delta").mode("overwrite").saveAsTable("wti_crude_raw")

brent_df.write.format("delta").mode("overwrite").saveAsTable("brent_crude_raw")

print("Bronze Layer Tables Created Successfully")

#ingest fii data to bronze layer

In [0]:
from pyspark.sql import functions as F

# --- 1. CONFIGURATION ---
fii_path = "/Volumes/iran_israel_capstone_project/bronze/landing_zone/fii_data/Merged_FII_Data.csv"

# --- 2. INGESTION ---
df_fii_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(fii_path)

# --- 3. TRANSFORMATION & MAPPING (DII REMOVED) ---
# We are only selecting the date and the three FII-specific columns.
# No DII columns are defined or selected here.
df_fii_mapped = df_fii_raw.select(
    F.to_date(F.col("DATE")).alias("date"),
    F.col("`FII EQUITY Net Purchase / Sales`").cast("float").alias("fii_net_buy_sell_cr"),
    F.col("`FII EQUITY Gross Purchase`").cast("float").alias("fii_gross_buy_cr"),
    F.col("`FII EQUITY Gross Sales`").cast("float").alias("fii_gross_sell_cr")
)

# --- 4. AUDIT COLUMNS ---
# Required for Bronze KPI Compliance 
df_fii_final = df_fii_mapped.withColumn("ingestion_timestamp", F.current_timestamp()) \
                            .withColumn("source_file", F.lit("Merged_FII_Data.csv"))

# --- 5. SAVE TO BRONZE TABLE ---
# Using overwrite mode to ensure the old schema (with the DII column) is replaced
df_fii_final.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fii_raw")

# --- 6. VERIFICATION ---
print("SUCCESS: Bronze Table 'fii_raw' updated. DII column has been removed.")
display(spark.table("iran_israel_capstone_project.bronze.fii_raw").limit(10))